# 04 — Prompt Engineering para RAG

## O prompt é o último quilômetro

Você indexou bem, fez retrieval eficiente, recuperou os documentos certos.
Agora vem o último passo: pedir ao LLM que gere uma resposta.

O prompt determina:
- **Fidelidade:** o LLM usa apenas o contexto fornecido ou "inventa" informações?
- **Formato:** a resposta é curta e direta ou longa e detalhada?
- **Atribuição:** o LLM cita as fontes?
- **Honestidade:** quando a resposta não está no contexto, o LLM admite ou alucina?

**Alucinação** é o maior risco de sistemas RAG. Um LLM bem instruído diz
"não encontrei essa informação nos documentos". Um LLM mal instruído inventa uma resposta convincente mas falsa.

Este notebook mostra a evolução do prompt: do básico ao production-grade.

> **Pré-requisito:** Ollama rodando com Llama 3.2 (`docker compose up -d` ou `ollama pull llama3.2`)

In [ ]:
import httpx, json

# Verificar se Ollama está disponível
try:
    r = httpx.get("http://localhost:11434/api/tags", timeout=3)
    models = [m["name"] for m in r.json().get("models", [])]
    LLM_OK = True
    print(f"Ollama disponível. Modelos: {models}")
    LLM_MODEL = "llama3.2" if any("llama3.2" in m for m in models) else models[0] if models else None
    print(f"Usando modelo: {LLM_MODEL}")
except Exception as e:
    LLM_OK = False
    LLM_MODEL = None
    print(f"Ollama não disponível ({e})")
    print("Os prompts serão mostrados mas não executados.")

In [ ]:
def gerar(prompt, model=LLM_MODEL, max_tokens=400):
    """Chama o Ollama e retorna a resposta."""
    if not LLM_OK:
        print("[Ollama offline — resposta simulada]")
        return "[Ollama não disponível]"
    try:
        r = httpx.post(
            "http://localhost:11434/api/generate",
            json={"model": model, "prompt": prompt, "stream": False,
                  "options": {"num_predict": max_tokens, "temperature": 0.1}},
            timeout=60
        )
        return r.json()["response"]
    except Exception as e:
        return f"[Erro: {e}]"

# Contexto de exemplo — simula o que o retrieval retornaria
contexto_chunks = [
    """HNSW (Hierarchical Navigable Small World) é um algoritmo de indexação vetorial.
Ele constrói um grafo hierárquico onde cada nó é conectado a seus vizinhos mais próximos.
A busca começa nas camadas superiores (poucos nós, conexões longas) e desce progressivamente.
Isso permite busca aproximada em O(log N) com ~95-99% de recall.""",

    """Parâmetros do HNSW:
- m: número de conexões por nó (padrão=16). Mais conexões = melhor recall, mais memória.
- ef_construct: candidatos considerados durante indexação (padrão=100).
- ef: candidatos na busca (padrão=128). Aumentar melhora recall sem precisar reindexar.""",

    """Qdrant é um banco de dados vetorial open-source escrito em Rust.
Suporta HNSW nativo, quantização escalar e por produto.
Permite combinar busca vetorial com filtros em metadados (payload) numa única query.""",
]

pergunta = "Quais são os parâmetros do HNSW e como eles afetam a performance?"
print("Pergunta:", pergunta)
print(f"\n{len(contexto_chunks)} chunks recuperados")

## 4.1 Prompt Básico

O prompt mais simples possível: forneça o contexto e faça a pergunta.

**Problema:** sem instrução explícita, o LLM tende a "completar o texto de forma coerente" —
o que significa potencialmente inventar informações que não estão nos documentos.

In [ ]:
PROMPT_BASICO = """Contexto:
{contexto}

Pergunta: {pergunta}
Resposta:"""

ctx = "\n\n".join(contexto_chunks)
prompt = PROMPT_BASICO.format(contexto=ctx, pergunta=pergunta)
print("=== PROMPT BÁSICO ===")
print(prompt[:300] + "..." if len(prompt) > 300 else prompt)
print()
resposta_basica = gerar(prompt)
print("RESPOSTA:", resposta_basica)

## 4.2 Prompt com Grounding Explícito

A solução para reduzir alucinação: instrua **explicitamente** o LLM a responder apenas com base no contexto.

A adição de frases como *"Responda SOMENTE com base nas informações fornecidas"* e
*"Se a resposta não estiver no contexto, diga 'Não encontrei essa informação'"* reduz drasticamente a taxa de alucinação.

**Por que funciona?** LLMs são treinados para seguir instruções.
Quando você explicita o comportamento esperado, o modelo ajusta seu output.
Instruções precisas ("SOMENTE com base no contexto") são mais efetivas que instruções vagas ("use o contexto").

In [ ]:
PROMPT_GROUNDED = """Você é um assistente especializado. Responda SOMENTE com base nos documentos abaixo.
Se a informação solicitada não estiver nos documentos, responda exatamente:
"Não encontrei essa informação nos documentos fornecidos."
Não adicione nenhuma informação que não esteja explicitamente nos documentos.

DOCUMENTOS:
{contexto}

PERGUNTA: {pergunta}

RESPOSTA:"""

prompt = PROMPT_GROUNDED.format(contexto=ctx, pergunta=pergunta)
print("=== PROMPT COM GROUNDING ===")
resposta_grounded = gerar(prompt)
print("RESPOSTA:", resposta_grounded)

## 4.3 Prompt com Citações

Para sistemas onde rastreabilidade é importante, pedir ao LLM que cite as fontes muda tudo.

**Por que citações são importantes:**
1. **Verificação:** o usuário pode confirmar na fonte original
2. **Confiança:** respostas com fontes parecem (e geralmente são) mais confiáveis
3. **Debug:** quando o sistema erra, você sabe qual chunk causou o problema
4. **Compliance:** em domínios regulados, rastreabilidade pode ser requisito

In [ ]:
PROMPT_COM_CITACOES = """Você é um assistente especializado. Responda SOMENTE com base nos documentos numerados abaixo.
Cite as fontes usando [1], [2], etc. ao final de cada afirmação.
Se a informação não estiver nos documentos, diga: "Não encontrei essa informação."

{docs_numerados}

PERGUNTA: {pergunta}

RESPOSTA (com citações):"""

docs_numerados = "\n\n".join(f"[{i+1}] {chunk}" for i, chunk in enumerate(contexto_chunks))
prompt = PROMPT_COM_CITACOES.format(docs_numerados=docs_numerados, pergunta=pergunta)
print("=== PROMPT COM CITAÇÕES ===")
resposta_citacoes = gerar(prompt)
print("RESPOSTA:", resposta_citacoes)

## 4.4 Teste de Alucinação

O teste mais importante: **o que acontece quando você pergunta algo que NÃO está nos documentos?**

Um sistema RAG bem construído deve dizer "não sei" de forma honesta.
Um sistema mal construído vai inventar uma resposta plausível — e isso é perigoso.

In [ ]:
pergunta_fora = "Qual é o preço do plano enterprise do Qdrant?"

print("=== TESTE DE ALUCINAÇÃO ===")
print(f"Pergunta: '{pergunta_fora}'")
print(f"(A resposta NÃO está nos documentos)\n")

print("--- Prompt Básico (propenso a alucinar) ---")
prompt_basico = PROMPT_BASICO.format(contexto=ctx, pergunta=pergunta_fora)
r1 = gerar(prompt_basico, max_tokens=150)
print(r1)

print("\n--- Prompt Grounded (deve recusar) ---")
prompt_grounded = PROMPT_GROUNDED.format(contexto=ctx, pergunta=pergunta_fora)
r2 = gerar(prompt_grounded, max_tokens=150)
print(r2)

### O que o teste revela?

A diferença entre os prompts é dramática em cenários de alucinação:

- **Prompt básico:** frequentemente gera respostas inventadas que *parecem* corretas
- **Prompt grounded:** força o LLM a admitir quando não sabe

**Insight crítico:** testar com perguntas "fora do escopo" é tão importante quanto testar com perguntas que o sistema deveria saber responder. Sistemas RAG em produção sempre recebem queries para as quais não têm resposta.

**Como testar sistematicamente:**
1. Conjunto A: queries com resposta conhecida (mede acurácia)
2. Conjunto B: queries cujas respostas não estão nos documentos (mede taxa de alucinação)

Ambos os conjuntos são necessários para uma avaliação completa.

## Resumo: Evolução do Prompt

| Versão | Alucinação | Rastreabilidade | Complexidade |
|--------|:---:|:---:|:---:|
| Básico | Alta | Nenhuma | Mínima |
| Com grounding | Baixa | Nenhuma | Baixa |
| Com citações | Baixa | Alta | Média |

**Template de produção recomendado:**
```
Você é um assistente especializado. Responda APENAS com base nos documentos abaixo.
Se a resposta não estiver nos documentos, diga: "Não encontrei essa informação."

DOCUMENTOS:
[1] {doc1}
[2] {doc2}

PERGUNTA: {query}

RESPOSTA (cite as fontes como [1], [2] etc.):
```

**Próximos passos:**
- [01 — Naive RAG Architecture](../04_rag_architectures/01_naive_rag_arch.html): como essas peças se combinam numa arquitetura de produção?